# BERT Fine-Tuning for Sentiment Classification

**ANLP Session 6 — Transformers & Language Models**

---

In the previous notebooks we explored BERT's internal mechanics — tokenisation, contextual embeddings, masked language modelling, and attention patterns. BERT was pre-trained on general text (BooksCorpus + Wikipedia) to develop rich language representations.

In practice, you rarely use BERT "raw". Instead, you **fine-tune** it on a labelled dataset for a specific task. This is the transfer learning workflow:

```mermaid
flowchart LR
    A["Pre-trained BERT<br/>(general NLU)"] --> B["Add task head"] --> C["Fine-tune on labelled data<br/>(a few epochs, small data)"] --> D["Task-specific model"]
```

This notebook covers two approaches:

1. **Pipeline inference** — use a ready-made fine-tuned model in one line
2. **Fine-tuning from scratch** — start from `bert-base-uncased` and train on the IMDB sentiment dataset

We use the [IMDB dataset](https://huggingface.co/datasets/imdb) — 50,000 movie reviews labelled as positive (1) or negative (0).

---

## Setup

We need:
- `transformers` — HuggingFace library (models, tokenisers, Trainer)
- `datasets` — HuggingFace dataset hub
- `torch` — PyTorch backend
- `scikit-learn`, `matplotlib`, `seaborn` — evaluation and visualisation

> **GPU note:** Fine-tuning BERT on 2,000 examples for 3 epochs takes roughly 5–15 minutes with a GPU and considerably longer on CPU. If you only have CPU, consider reducing `num_train_epochs` to 1 or `train_size` to 500 to keep the demo fast.

In [1]:
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from datasets import load_dataset
from transformers import (
    pipeline,
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import transformers
transformers.logging.set_verbosity_warning()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")

Device: cpu
PyTorch: 2.10.0+cpu
Transformers: 5.2.0


---

## Part 1 — Ready-Made Sentiment Analysis with Pipelines

HuggingFace's `pipeline()` is the quickest way to apply a fine-tuned model. Under the hood, it:
1. Downloads a model someone has already fine-tuned
2. Wraps tokenisation + inference into a single function call

We will use `nlptown/bert-base-multilingual-uncased-sentiment` — a BERT model fine-tuned on product reviews, returning a 1–5 star rating.

In [2]:
# Load a pre-fine-tuned sentiment pipeline — no training required
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

reviews = [
    "This movie was absolutely fantastic! The performances were stellar.",
    "The plot was dull and the pacing was unbearably slow.",
    "Decent film — enjoyable but nothing I'd watch twice.",
    "A masterpiece. One of the best films I have ever seen.",
    "Terrible acting, confusing plot, and a waste of two hours.",
]

print(f"{'Review':<60} {'Label':<12} {'Score'}")
print("-" * 85)
for review in reviews:
    result = sentiment_pipeline(review)[0]
    print(f"{review[:57]+'...':<60} {result['label']:<12} {result['score']:.3f}")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Review                                                       Label        Score
-------------------------------------------------------------------------------------
This movie was absolutely fantastic! The performances wer... 5 stars      0.934
The plot was dull and the pacing was unbearably slow....     2 stars      0.613
Decent film — enjoyable but nothing I'd watch twice....      3 stars      0.644
A masterpiece. One of the best films I have ever seen....    5 stars      0.973
Terrible acting, confusing plot, and a waste of two hours... 1 star       0.788


**Key point:** With two lines of code you get a fully functional sentiment classifier. This is the power of pre-trained + fine-tuned models.

But what if no existing model suits your task? That's when you fine-tune your own.

---

## Part 2 — Fine-Tuning BERT on the IMDB Dataset

Fine-tuning BERT for binary sentiment classification works like this:

1. **Load** `bert-base-uncased` — the 110M parameter pre-trained model
2. **Add a classification head** — a single linear layer on top of BERT's `[CLS]` output (2 output classes: positive / negative)
3. **Tokenise** the IMDB reviews
4. **Train** — the classification head learns from scratch; BERT's weights adjust slightly via backpropagation
5. **Evaluate** — accuracy and confusion matrix on a held-out test set

The pre-training gives BERT a deep understanding of English; fine-tuning teaches it what "positive" and "negative" mean for movie reviews.

### 2.1 Load the IMDB Dataset

In [3]:
# Load the full IMDB dataset from HuggingFace Hub
# 25,000 training examples, 25,000 test examples
dataset = load_dataset("imdb")
print(dataset)

# Label mapping: 0 = negative, 1 = positive
label_names = ["negative", "positive"]

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [4]:
import pandas as pd

# Preview the first few examples
sample_df = pd.DataFrame({
    "text": dataset["train"][:5]["text"],
    "label": [label_names[l] for l in dataset["train"][:5]["label"]],
})

# Truncate long text for display
sample_df["text"] = sample_df["text"].str[:200] + "..."
print(sample_df.to_string(index=False))

                                                                                                                                                                                                       text    label
I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ev... negative
"I Am Curious: Yellow" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that ... negative
If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches ... negative
This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements a

### 2.2 Tokenise the Dataset

BERT requires fixed-length input sequences of token IDs. We use `BertTokenizerFast` to:
- Split text into WordPiece tokens
- Truncate sequences longer than `max_length` tokens
- Pad shorter sequences to the same length within a batch
- Return the `attention_mask` — a binary mask so BERT ignores padding tokens

In [ ]:
# Load the tokeniser for bert-base-uncased
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize(examples):
    """
    Tokenise a batch of texts.
    - truncation=True: cut sequences longer than max_length (BERT's max is 512)
    - padding=True: pad within batch (Trainer handles per-batch padding via DataCollator)
    """
    return tokenizer(examples["text"], truncation=True, max_length=256)

# Apply tokenisation to the entire dataset in batches
tokenized_dataset = dataset.map(tokenize, batched=True)

# Prepare for PyTorch: set the format so the Dataset returns tensors
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("Tokenisation complete.")
print(f"Columns: {tokenized_dataset['train'].column_names}")

### 2.3 Create Training and Test Subsets

Fine-tuning on the full 25,000 example training set would take hours without a powerful GPU. Instead, we use a representative subset to demonstrate the workflow within a reasonable time.

Even with 2,000 training examples BERT typically reaches 85–90% accuracy — far better than training from scratch — because its weights already encode strong language understanding.

In [ ]:
TRAIN_SIZE = 2000   # reduce to 500 if running on CPU with no patience
TEST_SIZE  = 500

train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(TRAIN_SIZE))
test_dataset  = tokenized_dataset["test"].shuffle(seed=42).select(range(TEST_SIZE))

print(f"Training examples : {len(train_dataset)}")
print(f"Test examples     : {len(test_dataset)}")

### 2.4 Load the Model

`BertForSequenceClassification` is `bert-base-uncased` with a linear classification head attached. The head is randomly initialised — those weights will be learned during fine-tuning. BERT's body weights start from the pre-trained values.

In [ ]:
# num_labels=2: binary classification (negative / positive)
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

# Move to GPU if available
model = model.to(device)

# Count parameters
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters    : {total:,}")
print(f"Trainable parameters: {trainable:,}")
print()
print("The classification head (last layer):", model.classifier)

### 2.5 Train with HuggingFace Trainer

`TrainingArguments` controls the training loop; `Trainer` handles the rest:
- Batching and device management
- Gradient computation and optimiser steps
- Evaluation at the end of each epoch
- Logging

In [ ]:
def compute_metrics(pred):
    """Called by Trainer after each evaluation step to compute accuracy."""
    labels = pred.label_ids
    preds  = pred.predictions.argmax(-1)  # argmax over logits → class index
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}


training_args = TrainingArguments(
    output_dir="./results/bert-imdb",       # checkpoint and output directory
    eval_strategy="epoch",                   # evaluate after every epoch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,                      # standard fine-tuning LR for BERT
    weight_decay=0.01,                       # L2 regularisation
    warmup_ratio=0.1,                        # LR warmup over first 10% of steps
    logging_dir="./logs",
    logging_steps=50,
    report_to="none",                        # disable W&B / other integrations
    fp16=(device == "cuda"),                 # mixed precision for faster GPU training
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()

### 2.6 Evaluate — Confusion Matrix

A confusion matrix shows exactly where the model succeeds and where it confuses positive for negative (and vice versa). The diagonal cells are correct predictions.

In [ ]:
# Run inference on the test set and collect predictions
pred_output = trainer.predict(test_dataset)

true_labels = pred_output.label_ids
pred_labels = pred_output.predictions.argmax(-1)

# Overall accuracy
acc = accuracy_score(true_labels, pred_labels)
print(f"Test accuracy: {acc:.4f} ({acc*100:.1f}%)")
print()

# Detailed per-class metrics
print(classification_report(true_labels, pred_labels, target_names=label_names))

# Confusion matrix
cm = confusion_matrix(true_labels, pred_labels)

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_names, yticklabels=label_names,
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix — Fine-Tuned BERT\nIMDB Sentiment (Accuracy: {acc:.2%})")
plt.tight_layout()
plt.show()

---

## Part 3 — Using the Fine-Tuned Model

Once trained, you can use the model directly for inference. The snippet below wraps it in a `pipeline` just like Part 1 — but now the model has been adapted specifically to IMDB-style movie reviews.

In [ ]:
# Wrap the fine-tuned model and its tokeniser in an inference pipeline
imdb_pipeline = pipeline(
    "sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1,  # -1 = CPU
)

test_reviews = [
    "An absolute triumph of storytelling. I laughed, I cried, I stood up and clapped.",
    "Two hours I'll never get back. Predictable, clichéd, and poorly acted.",
    "Not bad, but not great either. Somewhere in the middle.",
]

print(f"{'Review':<65} {'Label':<12} {'Confidence'}")
print("-" * 90)
for review in test_reviews:
    result = imdb_pipeline(review, truncation=True, max_length=256)[0]
    print(f"{review[:62]+'...':<65} {result['label']:<12} {result['score']:.3f}")

---

## Part 4 — Saving and Loading the Model

After fine-tuning, save the model locally so you don't need to retrain it in future sessions.

In [ ]:
SAVE_DIR = "./results/bert-imdb-final"

# Save model weights and tokeniser together
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model saved to {SAVE_DIR}")

# --- To reload later: ---
# from transformers import pipeline
# imdb_pipeline = pipeline("sentiment-analysis", model=SAVE_DIR)
# result = imdb_pipeline("Your review here.")

---

## Summary

| Step | What happened |
|---|---|
| Pipeline inference | A one-line import of a ready-made fine-tuned model |
| Load dataset | IMDB: 25k train + 25k test, binary labels |
| Tokenise | `BertTokenizerFast` — truncate to 256 tokens, pad within batches |
| Load model | `bert-base-uncased` + 2-class classification head |
| Fine-tune | 3 epochs on 2,000 examples — BERT converges quickly |
| Evaluate | ~85–90% accuracy with a confusion matrix |
| Save | `model.save_pretrained()` stores weights for reuse |

**Why fine-tuning works so well with little data:** BERT's pre-training has already encoded the structure of English, word meanings, and long-range context. Fine-tuning simply teaches the classification head *which* patterns correspond to positive vs negative sentiment — a much easier task than learning language from scratch.

In Sessions 7 and 8 we will see this same pattern applied to other tasks — question answering, named entity recognition, and retrieval-augmented generation.